# sklearn model with tensorflow keras tuner

In [ ]:
from helper_func import *
import helper_func as hf
import nltk
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer
import string
from spellchecker import SpellChecker
from textblob import TextBlob
from multiprocessing import Pool
from tqdm import tqdm
import numpy as np
import pandas as pd
# Preprocessing
from nltk.tokenize import word_tokenize, sent_tokenize
import operator
from spellchecker import SpellChecker
from tqdm import tqdm  # Import tqdm
import re
import inflect
from wordsegment import load, segment
from nltk.corpus import words
word_list = set(words.words())

In [ ]:
train = pd.read_csv('data/train.csv')

In [ ]:
# time the run time

import time

start = time.time()

# Clean the train and test text data

train = hf.clean_text(train, col_name = 'full_text')


end = time.time()

print(f"Time taken to clean text: {end - start} seconds")

In [ ]:
train['misspelling_count'] = train['clean_text'].apply(count_misspellings)

In [ ]:
train = add_text_features(train, 'full_text', status= 'Pre')

In [ ]:
text_col = 'clean_text'   # 'segmented_text'

In [ ]:
glove_path = '/home/laptop/github/kaggle/scoring/data/glove-840B-300d.txt'
paragran_path = '/home/laptop/github/kaggle/scoring/data/paragram-300-sl999.txt'
fastetxt_path = '/home/laptop/github/kaggle/scoring/data/wiki-news-1M-300d.vec'


# Rebuild and check vocab after cleaning contractions

train, glove, paragram, fastetxt = embedding_checks(train, glove_path, paragran_path, fastetxt_path, col_name= text_col)

In [ ]:
misspellings = []

oov = glove + paragram + fastetxt

for word, _ in oov:

    misspellings.append(word)

    misspellings = list(set(misspellings))

print(f"Number of misspelled words: {len(misspellings)}")

# print(f"Misspelled words: {misspellings}")

In [ ]:
# Example usage
corrected_words, uncorrected_words = main(misspellings)

In [ ]:
# Assuming df is your DataFrame and 'clean_text' is the column you want to correct

correction_dict = dict(corrected_words)

train['corrected_text'] = train[text_col].apply(lambda x: apply_corrections_to_text(x, correction_dict))


In [ ]:
print(f"Corrected words: {len(corrected_words)}")
print(f"Uncorrected words: {len(uncorrected_words)}")

In [ ]:
# Apply the parallelization

train = parallelize_dataframe(train, apply_segmentation)

train.head()


In [ ]:
# text_col = 'segmented_text'  


# glove_path = '/home/laptop/github/kaggle/scoring/data/glove-840B-300d.txt'
# paragran_path = '/home/laptop/github/kaggle/scoring/data/paragram-300-sl999.txt'
# fastetxt_path = '/home/laptop/github/kaggle/scoring/data/wiki-news-1M-300d.vec'


# # Rebuild and check vocab after cleaning contractions

# train, glove, paragram, fastetxt = embedding_checks(train, glove_path, paragran_path, fastetxt_path, col_name= text_col)


# misspellings = []

# oov = glove + paragram + fastetxt

# for word, _ in oov:

#     misspellings.append(word)

#     misspellings = list(set(misspellings))

# print(f"Number of misspelled words: {len(misspellings)}")

# # print(f"Misspelled words: {misspellings}")


# # Example usage
# corrected_words, uncorrected_words = main(misspellings)

# # Assuming df is your DataFrame and 'clean_text' is the column you want to correct

# correction_dict = dict(corrected_words)

# train['corrected_text'] = train[text_col].apply(lambda x: apply_corrections_to_text(x, correction_dict))

# print(f"Corrected words: {len(corrected_words)}")
# print(f"Uncorrected words: {len(uncorrected_words)}")


In [ ]:
train_essays, validation_essays = custom_train_validation_split(train, test_size=0.25, random_state=42)

In [ ]:
train_essays, _ = preprocess_data(train_essays)

In [ ]:
validation_essays, _ = preprocess_data(validation_essays) #, tfidf_vectorizer=tfidf_vectorizer)

In [ ]:
train_essays.columns

In [ ]:
STATUS = 'Pre'

if STATUS == 'Post':
       
       drop_cols = ['full_text', 'lowered', 'clean_text',
        'Pre_tokens', 'Pre_sentences'
       , 'Pre_pos_tags','corrected_text', 'segmented_text',
       'Post_tokens', 'Post_sentences', 'Post_pos_tags']

else:
       drop_cols = ['full_text', 'lowered', 'clean_text',
        'Pre_tokens', 'Pre_sentences'
       , 'Pre_pos_tags','corrected_text', 'segmented_text',]

training = train_essays.copy()
validation = validation_essays.copy()

training.drop(columns=drop_cols, inplace= True)
validation.drop(columns=drop_cols, inplace= True)



In [ ]:
feature_cols = []

for col in training.columns:
    if (col != 'essay_id') and (col != 'score'):
        feature_cols.append(col) 


# Select relevant columns (replace with actual column names)

training_labels = training['score']

validation_labels = validation['score']

train_features = training[feature_cols].values

val_features = validation[feature_cols].values

# test_features = test[feature_cols].values


# Standardize the features if needed

scaler = StandardScaler()


train_features = scaler.fit_transform(train_features)

val_features = scaler.transform(val_features)


target_scaler = StandardScaler()

# training_labels = target_scaler.fit_transform(training_labels.values.reshape(-1, 1)).flatten()

# validation_labels = target_scaler.transform(validation_labels.values.reshape(-1, 1)).flatten()




# Optionally, convert back to DataFrame

train_labels = pd.DataFrame(training_labels, columns=['score'],
                             index=training_labels.index)

val_labels = pd.DataFrame(validation_labels, columns=['score'], 
                          index=validation_labels.index)

import pickle


with open('data/scalers/sklearn_scaler.pkl', 'wb') as f:
    pickle.dump(scaler, f)
    


In [ ]:
scaler_path = 'data/scalers/sklearn_scaler.pkl'

# Check if the file has been written correctly and is not empty
import os

if os.path.getsize(scaler_path) > 0:
    print(f"Scaler saved successfully in {scaler_path}.")
else:
    print(f"Failed to save scaler to {scaler_path}. File is empty.")

In [ ]:
import tensorflow as tf

train_class = tf.data.Dataset.from_tensor_slices((train_features, 
                                                training_labels)).shuffle(len(train_features)).batch(32)

val_class = tf.data.Dataset.from_tensor_slices((val_features, 
                                              validation_labels)).batch(32)


In [ ]:
import keras_tuner
from sklearn import ensemble
from sklearn import linear_model
from sklearn import model_selection
from sklearn import metrics
from sklearn.metrics import make_scorer, cohen_kappa_score

def build_model(hp):
    """
    Builds a machine learning model based on hyperparameters.
    
    Parameters:
    hp : HyperParameters
        Hyperparameters for tuning the model.
    
    Returns:
    model : An instance of a Scikit-learn model.
    """
    model_type = hp.Choice('model_type', ['random_forest', 'ridge'])
    if model_type == 'random_forest':
        model = ensemble.RandomForestClassifier(
            n_estimators=hp.Int('n_estimators', 10, 50, step=10),
            max_depth=hp.Int('max_depth', 3, 10),
            min_samples_split=hp.Int('min_samples_split', 2, 10),
            min_samples_leaf=hp.Int('min_samples_leaf', 1, 10),
            criterion=hp.Choice('criterion', ['gini', 'entropy']),
            class_weight=hp.Choice('class_weight', ['balanced', 'balanced_subsample']),
            max_samples=hp.Float('max_samples', 0.1, 1.0, sampling='log'))
    else:
        model = linear_model.RidgeClassifier(
            alpha=hp.Float('alpha', 1e-3, 1, sampling='log'))

    return model

def quadratic_weighted_kappa_scorer(y_true, y_pred):
    """
    Compute the Quadratic Weighted Kappa (QWK), also known as Cohen's kappa.
    
    Parameters:
    y_true : array-like of shape (n_samples,)
        True labels.
    y_pred : array-liimport keras_tuner
from sklearn import ensemble
from sklearn import datasets
from sklearn import linear_model
from sklearn import metrics
from sklearn import model_selection

def build_model(hp):
  model_type = hp.Choice('model_type', ['random_forest', 'ridge'])
  if model_type == 'random_forest':
    model = ensemble.RandomForestClassifier(
        n_estimators=hp.Int('n_estimators', 10, 50, step=10),
        max_depth=hp.Int('max_depth', 3, 10))
  else:
    model = linear_model.RidgeClassifier(
        alpha=hp.Float('alpha', 1e-3, 1, sampling='log'))
  return model

tuner = keras_tuner.tuners.SklearnTuner(
    oracle=keras_tuner.oracles.BayesianOptimizationOracle(
        objective=keras_tuner.Objective('score', 'max'),
        max_trials=100),
    hypermodel=build_model,
    scoring=metrics.make_scorer(metrics.accuracy_score),
    cv=model_selection.StratifiedKFold(10),
    directory='.',
    project_name='my_project')ke of shape (n_samples,)
        Predicted labels.
    
    Returns:
    score : float
        Quadratic Weighted Kappa score.
    """
    return cohen_kappa_score(y_true, y_pred, weights='quadratic')

# Wrapping QWK as a custom scorer for model evaluation
qwk_scorer = make_scorer(quadratic_weighted_kappa_scorer)

# Tuner configuration
tuner = keras_tuner.tuners.SklearnTuner(
    oracle=keras_tuner.oracles.BayesianOptimizationOracle(
        objective=keras_tuner.Objective('score', 'max'),
        max_trials=100),
    hypermodel=build_model,
    scoring=qwk_scorer,  # Use the QWK scorer
    cv=model_selection.StratifiedKFold(10),
    directory='.',
    project_name='data/sklearn',
    overwrite=True)


tuner.search(train_features, train_labels)

best_model = tuner.get_best_models(num_models=1)[0]

In [ ]:
best_model.fit(train_features, train_labels)

In [ ]:
predictions = best_model.predict(val_features)

# predictions = target_scaler.inverse_transform(predictions.reshape(-1, 1)).flatten()


In [ ]:
# confusion matrix

from sklearn.metrics import confusion_matrix

confusion_matrix(val_labels, predictions)

# classification report

from sklearn.metrics import classification_report

print(classification_report(validation_labels, predictions))


In [ ]:
# cohens kappa

from sklearn.metrics import cohen_kappa_score

cohen = cohen_kappa_score(val_labels, predictions)

# quadratic weighted kappa

from sklearn.metrics import cohen_kappa_score

quadratic = cohen_kappa_score(val_labels, predictions, weights='quadratic')

print(f"Cohen's Kappa: {cohen}")
print(f"Quadratic Weighted Kappa: {quadratic}")

In [ ]:
from joblib import dump, load

dump(best_model, 'data/sklearn/random_forest.joblib') 


In [ ]:
# forest_model = load('random_forest.joblib') 